In [19]:
import os
print(os.getcwd())

/Users/revathy_santhosh/rag-project


In [20]:
import os
from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel

load_dotenv(dotenv_path="/Users/revathy_santhosh/rag-project/.env")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def ask(messages, model="openai/gpt-oss-20b"):
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [21]:
review = "The battery life on this laptop is incredible, but the screen hinge feels flimsy."

zero_shot_prompt = [
    {"role": "user", "content": f"Classify the sentiment of this review as Positive, Negative, or Mixed:\n\n{review}"}
]
print(ask(zero_shot_prompt))

Mixed


In [22]:
few_shot_prompt = [
    {"role": "system", "content": "Classify each support message into exactly one category: Billing, Technical, or General."},
    {"role": "user", "content": "I was charged twice for my subscription this month."},
    {"role": "assistant", "content": "Billing"},
    {"role": "user", "content": "The app crashes every time I try to upload a photo."},
    {"role": "assistant", "content": "Technical"},
    {"role": "user", "content": "What are your office hours?"},
    {"role": "assistant", "content": "General"},
    # New message to classify — not labeled, this is what we're testing
    {"role": "user", "content": "My invoice shows an extra $15 fee I don't recognize."},
]

print(ask(few_shot_prompt))

Billing


In [23]:
cot_prompt = [
    {"role": "system", "content": "You are a support ticket triager. Think through the impact and urgency step by step, then give your final line in the exact format: 'Priority: <Low|Medium|High>'."},
    {"role": "user", "content": "Ticket: 'Our entire finance team (12 people) can't log in since this morning — they use this tool daily to close month-end books, and the deadline is tomorrow.'"}
]

print(ask(cot_prompt))

Priority: High


In [24]:
class TicketExtraction(BaseModel):
    category: str
    priority: str
    summary: str

structured_prompt = [
    {"role": "system", "content": (
        "Extract structured info from the support ticket. "
        "Respond with ONLY valid JSON, no extra text, with exactly these keys: "
        "category (Billing, Technical, or General), priority (Low, Medium, or High), "
        "summary (one sentence)."
    )},
    {"role": "user", "content": "Ticket: 'Our entire finance team (12 people) can't log in since this morning — they use this tool daily to close month-end books, and the deadline is tomorrow.'"}
]

raw = ask(structured_prompt)
print("Raw output:", raw)

ticket = TicketExtraction.model_validate_json(raw)
print("\nParsed:", ticket)

Raw output: {
  "category": "Technical",
  "priority": "High",
  "summary": "Finance team of 12 members unable to log in since morning, threatening month-end close deadline tomorrow."
}

Parsed: category='Technical' priority='High' summary='Finance team of 12 members unable to log in since morning, threatening month-end close deadline tomorrow.'


In [25]:
class OrderRequest(BaseModel):
    sender_name: str
    company: str
    item: str
    quantity: int
    deadline: str

rcft_prompt = [
    {"role": "system", "content": (
        "ROLE: You are a sales operations assistant who extracts order details from customer emails.\n"
        "CONTEXT: Customers email in informally, often burying the actual order details in pleasantries.\n"
        "TASK: Read the email and extract the order details.\n"
        "FORMAT: Respond with ONLY valid JSON, no extra text, with keys: "
        "sender_name, company, item, quantity (integer), deadline."
    )},
    {"role": "user", "content": (
        "Hey there! Hope you're doing well. This is Maria from Northwind Logistics — "
        "we worked together last quarter on the warehouse shipment. Anyway, we're gearing up "
        "for the holiday season and need 500 units of Widget X sent over. Would love to have "
        "them in our Chicago warehouse by next Friday if at all possible. Thanks so much!"
    )}
]

raw = ask(rcft_prompt)
print("Raw output:", raw)

order = OrderRequest.model_validate_json(raw)
print("\nParsed:", order)

Raw output: {"sender_name":"Maria","company":"Northwind Logistics","item":"Widget X","quantity":500,"deadline":"next Friday"}

Parsed: sender_name='Maria' company='Northwind Logistics' item='Widget X' quantity=500 deadline='next Friday'
